# 🏠 AI Project 1: Support Vector Regression (SVR)
## Kathmandu Valley House Price Prediction (Nepal Real Estate)

### Objective:
Predict property valuation in the Kathmandu Valley (in Nepalese Rupees, NPR) using real listing data and Support Vector Regression (SVR).

### SVR Mathematical Foundation:
Support Vector Regression finds a function $f(x) = \langle w, \phi(x) \rangle + b$ that has at most $\varepsilon$ deviation from the actual target targets $y_i$ for all training data, while remaining as flat as possible.

$$\min_{w, b, \xi, \xi^*} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n (\xi_i + \xi_i^*)$$
$$\text{subject to } \begin{cases} y_i - \langle w, \phi(x_i) \rangle - b \le \varepsilon + \xi_i \\ \langle w, \phi(x_i) \rangle + b - y_i \le \varepsilon + \xi_i^* \\ \xi_i, \xi_i^* \ge 0 \end{cases}$$

Where:
- $\varepsilon$ determines the width of the insensitive tube.
- $C$ is the box constraint penalizing errors outside the $\varepsilon$-tube.
- $\phi(x)$ is the non-linear mapping induced by kernels like RBF ($K(x, x') = \exp(-\gamma \|x-x'\|^2)$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")

## 1. Load and Explore Nepal Housing Dataset

In [ ]:
df = pd.read_csv('data/nepal_house_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='Area_SqFt', y='Price_Lakhs', hue='City', alpha=0.7, ax=axes[0])
axes[0].set_title('Land Area vs Price (Lakhs NPR)', fontweight='bold')

sns.boxplot(data=df, x='City', y='Price_Lakhs', palette='Set2', ax=axes[1])
axes[1].set_title('Price Distribution by City in Nepal', fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Train/Test Split

In [ ]:
num_features = ['Area_SqFt', 'Bedroom', 'Bathroom', 'Floors', 'Parking', 'Road_Width_Ft']
cat_features = ['City', 'Face']

X = df[num_features + cat_features]
y_lakhs = df['Price_Lakhs'].values
y_log = np.log1p(y_lakhs)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

## 4. Train SVR with Different Kernels (Linear, RBF, Poly)

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    model = Pipeline([
        ('prep', preprocessor),
        ('svr', SVR(kernel=kernel, C=10.0, epsilon=0.1))
    ])
    model.fit(X_train, y_train)
    pred_log = model.predict(X_test)
    pred = np.expm1(pred_log)
    test = np.expm1(y_test)
    print(f"Kernel: {kernel.upper():<7} | R2: {r2_score(test, pred):.4f} | MAE: {mean_absolute_error(test, pred):.2f} Lakhs")

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'svr__C': [1.0, 10.0, 50.0, 100.0],
    'svr__epsilon': [0.01, 0.05, 0.1, 0.2],
    'svr__gamma': ['scale', 'auto', 0.01, 0.1]
}

grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('svr', SVR(kernel='rbf'))]),
    param_grid, cv=5, scoring='r2', n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)
best_svr = grid.best_estimator_

## 6. Evaluation & Visualizations

In [ ]:
y_pred_best = np.expm1(best_svr.predict(X_test))
y_test_orig = np.expm1(y_test)

print(f"Test R² Score : {r2_score(y_test_orig, y_pred_best):.4f}")
print(f"Test MAE (Lakhs): {mean_absolute_error(y_test_orig, y_pred_best):.2f} Lakhs NPR")
print(f"Test RMSE (Lakhs): {np.sqrt(mean_squared_error(y_test_orig, y_pred_best)):.2f} Lakhs NPR")

plt.figure(figsize=(8, 7))
plt.scatter(y_test_orig, y_pred_best, alpha=0.6, color='#1f77b4', edgecolors='k')
max_val = max(y_test_orig.max(), y_pred_best.max())
plt.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Ideal Fit')
plt.xlabel('Actual Price (Lakhs NPR)')
plt.ylabel('SVR Predicted Price (Lakhs NPR)')
plt.title('Kathmandu House Prices: Actual vs SVR Predicted', fontweight='bold')
plt.legend()
plt.show()

## 7. Interactive Single Property Price Predictor

In [ ]:
def estimate_price(city, aana, bed, bath, floors, parking, road_ft, face):
    sample = pd.DataFrame([{
        'Area_SqFt': aana * 342.25,
        'Bedroom': bed,
        'Bathroom': bath,
        'Floors': floors,
        'Parking': parking,
        'Road_Width_Ft': road_ft,
        'City': city,
        'Face': face
    }])
    pred_lakhs = np.expm1(best_svr.predict(sample)[0])
    print(f"Estimated Price for {aana} Aana house in {city}: {pred_lakhs:.2f} Lakhs (NPR {pred_lakhs*100000:,.0f})")

estimate_price(city='Kathmandu', aana=4.0, bed=5, bath=3, floors=2.5, parking=1, road_ft=14.0, face='East')